# Active Learning using Laplax

In this example notebook, we demonstrate how laplax can be used to learn a deep neural network actively. \
It is based on the article *Information-Based Objective Functions for Active Data Selection* by David MacKay[1].

Active learning means to pick the datapoints used for training iteratively and in a smart manner, 
maximizing the information they give the network. \
We start by implementing the four core mechanics necessary to do active learning:
1) Sample a target given an x-value from the true function
2) Train the model using a given dataset of points
3) Compue the posterior covariance of the model
4) Find the most informative datapoint using a heuristic based on the posterior covariance

Part 1) and 2) are identical to what you would do in passive learning, i.e. normally \
Part 3) is where we are going to use laplax. \
For part 4), we are going to showcase the different heuristics introduced by MacKay.

Active learning then iterates through these steps in order to learn the function in a data-efficient manner. \
This is especially useful when labelling data is expensive, e.g. when it has to be labelled manually by experts
or acquired through a physics experiment.

[^1]: [David J. C. MacKay, *Information-Based Objective Functions for Active Data Selection*, 1992]

In [ ]:
from functools import partial

import ipywidgets as widgets
import jax
import loguru
import optax
from flax import nnx
from helper import DataLoader
from IPython.display import HTML
from jax import numpy as jnp
from jax import random, vmap
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from plotting import plot_data_and_uncertainty_around_prediction, plot_model_comparison

from laplax.curv import create_ggn_mv, create_posterior_fn
from laplax.eval import evaluate_for_given_prior_arguments
from laplax.eval.calibrate import optimize_prior_prec
from laplax.eval.metrics import nll_gaussian
from laplax.eval.pushforward import (
    lin_pred_mean,
    lin_pred_std,
    lin_setup,
    set_lin_pushforward,
    set_posterior_gp_kernel,
)

seed = 2392385

## Problem setup

We first choose a function that we want to learn, for now a simple 1D to 1D function, the sinus cardinalis. \
We define a function that computes the value of the function at a given point, and adds Gaussian measurement noise.

You can vary the noise variance using the slider widget below. There will be more choices to play around with throughout this notebook. \
Remember to execute all cells below after changing the value of a widget.

In [ ]:
var_widget = widgets.FloatLogSlider(
    value=0.01, base=10, min=-4, max=-1, step=0.5, description="Variance"
)
display(var_widget)

In [ ]:
sample_variance = var_widget.value
print("Sample variance: ", sample_variance)


def sample_target(x, key, sample_variance=0.0005):
    """Sample a target (label) for a given datapoint x.

    Args:
        x: x-value for which to sample a label
        key: PRNGKey to use for sampling
        sample_variance: Strength of added noise

    Returns:
        $y = f(x) + eps$ where $f$ is the sinc function
        and eps is Gaussian noise with mean zero and variance given by 'sample_variance'
    """
    x = x.squeeze()  # Assume one-dimensional x values
    y_true = jnp.sinc(x)
    noise = random.normal(key, y_true.shape) * jnp.sqrt(sample_variance)
    return y_true + noise


# Function without noise
def true_function(xs):
    key = random.key(seed)
    keys = random.split(key, len(xs))

    function = partial(sample_target, sample_variance=0.0)
    return vmap(function)(xs, keys)


# Initial dataset
x = jnp.concatenate((jnp.linspace(0.2, 2, 2), jnp.linspace(3.5, 5, 2)))[:, None]
x = x.astype(float)
n_initial_datapoints = x.shape[0]

key = random.key(seed)
keys = random.split(key, len(x))

sample = partial(sample_target, sample_variance=sample_variance)
y = vmap(sample)(x, keys)[:, None]

dataloader = DataLoader(x, y, batch_size=10)

We now have a preliminary dataset of just four points, sampled from the function.

## Model definition

Next, we define our deep neural network and its training loop.\
Here, we use a network of 4 fully connected layers with a hidden dimension of 32.

In [ ]:
class Model(nnx.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, rngs):
        self.linear1 = nnx.Linear(in_channels, hidden_channels, rngs=rngs)
        self.linear2 = nnx.Linear(hidden_channels, hidden_channels, rngs=rngs)
        self.linear3 = nnx.Linear(hidden_channels, hidden_channels, rngs=rngs)
        self.linear4 = nnx.Linear(hidden_channels, out_channels, rngs=rngs)

    def __call__(self, x):
        x = nnx.tanh(self.linear1(x))
        x = nnx.tanh(self.linear2(x))
        x = nnx.tanh(self.linear3(x))
        return self.linear4(x)


@nnx.jit
def train_step(model, optimizer, x, y):
    def loss_fn(model):
        y_pred = model(x)
        return jnp.sum((y_pred - y) ** 2)

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)

    return loss


model = Model(in_channels=1, hidden_channels=32, out_channels=1, rngs=nnx.Rngs(seed))

params = nnx.state(model)
total_params = sum(p.size for p in jax.tree.leaves(params))
print(f"Total number of parameters: {total_params}")

## Training loop

We define the training loop and train our model on the tiny starting dataset.

In [ ]:
def train_model(model, dataloader, n_epochs=1000, lr=1e-3):
    """Trains the given model on the data.

    Args:
        model: nnx.Module that represents the model, can be pretrained
        dataloader: Data on which to train
        n_epochs: Number of epochs to train for
        lr: learning rate for optimizer

    Returns:
        Trained model
    """
    optimizer = nnx.Optimizer(model, optax.adam(lr))

    for epoch in range(n_epochs):
        for x_batch, y_batch in dataloader:
            loss = train_step(model, optimizer, x_batch, y_batch)

        if epoch % 100 == 0 and epoch != 0:
            print(f"[epoch {epoch}]: loss: {loss:.4f}")
    print(f"Final loss: {loss:.4f}")
    return model


n_initial_epochs = 400

model = train_model(model, dataloader, n_epochs=n_initial_epochs)

Let's now visualize what we have so far:

In [ ]:
# Trained model visualization

x_pred = jnp.linspace(0.0, 5.9, 100)[:, None]

y_true = true_function(x_pred)
y_pred = model(x_pred)

fig, ax = plt.subplots(figsize=(10, 5))
plot_data_and_uncertainty_around_prediction(ax, x_pred, y_pred, y_true, dataloader)
plt.show()

The plot visualizes the true function's and datapoints difference to the prediction. This visualization is chosen such that later, we can visualize the information criteria nicely around the prediction.

The model fits the four datapoints well, but it does of course not match the true function well, because it has not seen enough data yet.

This concludes steps 1) and 2). Next, we turn to step 3), getting a posterior covariance kernel. \
This will give us the necessary information to make decisions about which datapoint to choose next.

The posterior covariance kernel is a function that takes two x-values and returns the estimated covariance between them given a (probabilistic) model. \
Since our deep neural network is not probabilistic, we need to add this probabilistic functionality. This is exactly what Laplax is designed to do. \
We use it to do a Laplace approximation in the weight space and push it forward into the output space.

## Laplax setup

First, we can choose how to approximate the curvature matrix of the network parameters.

By default, we choose the full curvature matrix. This is of course the most accurate, but most expensive option. \
Since our network is quite small, the full matrix would have only $2209^2 \text{ parameters} \cdot 4 \text{ byte} = 19.5 \text{ MB}$. \
Also, laplax never instantiates the full matrix, but performs the downstream calculations in a memory-efficient manner.

You can try one of the low-rank methods or even a diagonal approximation to see how this speeds up the computation.

In [ ]:
lib_dropdown = widgets.Dropdown(
    options=["full", "diagonal", "lanczos", "lobpcg"],
    value="full",
    description="Curv. est.:",
)
display(lib_dropdown)

In [ ]:
print(f"Curvature will be estimated using a {lib_dropdown.value} approximation.")
curv_type = lib_dropdown.value
low_rank_args = {
    "key": random.key(20),
    "rank": 50,
    "mv_jit": True,
}
curv_args = {} if curv_type in {"full", "diagonal"} else low_rank_args

We start by implementing some functions that will ultimately yield the posterior covariance kernel computed from the model.

In [ ]:
def split(model):
    """Split an nnx module into parameters and parameter-agnostic function.

    Args:
        model: nnx.module to split.

    Returns:
        Tuple of callable function taking model input and parameters,
        and model parameters.
    """
    graph_def, params = nnx.split(model)

    def model_fn(input, params):
        return nnx.call((graph_def, params))(input)[0]

    return model_fn, params


def get_posterior_fn(model, data):
    trainset = {"input": data.X, "target": data.y}
    model_fn, params = split(model)

    ggn_mv = create_ggn_mv(
        model_fn,
        params,
        trainset,
        loss_fn="mse",
    )

    return create_posterior_fn(
        curv_type=curv_type,
        mv=ggn_mv,
        layout=params,
        **curv_args,
    )


def get_posterior_covariance_kernel(model, posterior_fn, prior_prec):
    model_fn, params = split(model)
    gp_kernel, _ = set_posterior_gp_kernel(
        model_fn=model_fn,
        mean=params,
        posterior_fn=posterior_fn,
        prior_arguments={"prior_prec": prior_prec},
        dense=True,  # If dense = False, returns a slower kernel-vector product.
        output_layout=1,
    )

    def vectorized_laplace_kernel(a, b):
        return jnp.vectorize(gp_kernel, signature="(d),(d)->(j,j)")(a, b)[..., 0]

    return vectorized_laplace_kernel

To compute the posterior kernel function, we need a prior for the precision. \
As a best guess, we choose the inverse of the measurement variance.

In [ ]:
prior_prec = 1.0 / sample_variance

posterior_fn = get_posterior_fn(model, dataloader)
kernel = get_posterior_covariance_kernel(model, posterior_fn, prior_prec)

# This cell executes near instantly, as no actual computation is performed yet.
# Everything is evaluated lazily.

Now, let's get into what to do with the obtained kernel, approaching step 4) of the active learning protocol. \

The question we need to answer here is the following: \
Where do we need to sample next in order to maximize the information the learning algorithm gets about the parameters from the sampled point?

The answer is really intuitive, as MacKay shows in chapter 3: We sample where the uncertainty of the model is largest.

Thankfully, it is straight-forward to calculate the uncertainty from the kernel:

$$\text{std}(x) = \sqrt{k(x,x)}$$

> [!NOTE]
> There is a simpler way to get the model's uncertainty without using the full posterior kernel,
> by using the laplax.eval.pushforward.set_prob_predictive() function.
> We use the posterior kernel here, because we need its functionality for the later data acquisition rules.

In [ ]:
def get_uncertainty_from_kernel(kernel, x_pred):
    result = kernel(x_pred, x_pred).squeeze()
    return jnp.sqrt(result)


uncertainty = get_uncertainty_from_kernel(kernel, x_pred)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_data_and_uncertainty_around_prediction(
    ax, x_pred, y_pred, y_true, dataloader, uncertainty
)
plt.show()

We see that the computed uncertainty is very large. Ideally, we would want it to be indicative of the standard deviation of the datapoints to the mean prediction: \
For a well-calibrated model, the residuals are Gaussian with a standard deviation that is equal to the models uncertainty. \
Here however, the model is very underconfident.

To counter this, we can calibrate the model on the data by tuning the prior precision. \
We do this by grid searching a range of precision values and evaluating a Gaussian negative log likelihood objective for the data points under the model uncertainty. \
This is also something laplax can do for us.

## Prior precision calibration

In [ ]:
def calibrate_prior_precision(data, model, posterior_fn, grid_params):
    """Calibrate the prior precision.

    Args:
        data: dataloader to use for calibration
        model: nnx.Module
        posterior_fn: posterior function of the model,
        precomputed by laplax
        grid_params: dict of parameters for grid search

    Returns:
        Calibrated prior precision.
    """
    calibration_batch = {"input": data.X, "target": data.y}
    model_fn, params = split(model)

    prob_predictive = partial(
        set_lin_pushforward,
        model_fn=model_fn,
        mean_params=params,
        posterior_fn=posterior_fn,
        pushforward_fns=[
            lin_setup,
            lin_pred_mean,
            lin_pred_std,
        ],
    )

    @jax.jit
    def nll_objective(prior_arguments, batch):
        return evaluate_for_given_prior_arguments(
            prior_arguments=prior_arguments,
            data=batch,
            set_prob_predictive=prob_predictive,
            metric=nll_gaussian,
        )

    # Optimize via grid search
    guess_magnitude = jnp.log10(grid_params["current_guess"])
    prior_prec = optimize_prior_prec(
        objective=partial(nll_objective, batch=calibration_batch),
        log_prior_prec_min=guess_magnitude - grid_params["magnitudes_to_search"] / 2.0,
        log_prior_prec_max=guess_magnitude + grid_params["magnitudes_to_search"] / 2.0,
        grid_size=grid_params["grid_size"],
    )
    return prior_prec


grid_params = {
    "current_guess": 1.0 / sample_variance,  # Best guess before calibration
    "magnitudes_to_search": 6,
    "grid_size": 50,
}

prior_prec = calibrate_prior_precision(dataloader, model, posterior_fn, grid_params)

print("Prior precision: ", prior_prec)

We plot the learned network again, this time with the calibrated uncertainty.

In [ ]:
y_mean = model(x_pred)
kernel = get_posterior_covariance_kernel(model, posterior_fn, prior_prec)
y_std = get_uncertainty_from_kernel(kernel, x_pred)

fig, ax = plt.subplots(figsize=(10, 5))
plot_data_and_uncertainty_around_prediction(
    ax, x_pred, y_mean, y_true, dataloader, y_std
)
plt.show()

Now, the uncertainty resembles the magnitude of the errors our model makes much better.

## Maximizing total information gain

Implementing the data acquisition rule for maximal total information gain is now very simple: \
We choose the point where the uncertainty is largest. \
It is important to note that calibration can actually influence this choice, as changing the prior precision is not just a rescaling of the uncertainty.

In [ ]:
def find_maximum(x_pred, criterium):
    """Find the point in x_pred where criterium is maximal.

    Args:
        x_pred: Array of x valuesof which uncertainty is known
        criterium: The criterium values to maximize

    Returns:
        x-value with largest criterium value
    """
    next_index = jnp.argmax(criterium)
    return x_pred[next_index]


def maximize_total_information_gain(kernel, x_pred):
    """Find point where the total information gain is maximal.

    Args:
        kernel: Posterior covariance kernel of the model
        x_pred: Candidate points

    Returns:
        Point from x_pred where posterior covariance is maximal,
        and hence, total information gain is maximal.
    """
    uncertainty = get_uncertainty_from_kernel(kernel, x_pred)
    return find_maximum(x_pred, uncertainty)


next_datapoint = maximize_total_information_gain(kernel, x_pred)

## Active learning loop

Now that we have implemented and demonstrated all four steps, we can implement the full active learning loop,\
iteratively sampling the next datapoint, adding it to the trainset, continuing training for 100 epochs,\
recomputing the uncertainty, and finding the next best location. \
We also recalibrate the model in every step, as the calibration depends on the number of datapoints we have. \
We again calibrate by grid search, this time with a small grid around the previous value.

In [ ]:
epochs_per_learning_round = 100
learning_rounds = 16


def active_learning_loop(
    model, next_datapoint, dataloader, prior_prec, learning_rounds
):
    key = random.key(21780)
    keys = random.split(key, learning_rounds)

    plot_data = []

    loguru.logger.disable("laplax.eval.calibrate")  # Disable logging for grid search

    for i, key in enumerate(keys):
        print(f"Active learning round {i + 1}")
        # 1) Sample new datapoint
        next_target = sample_target(
            next_datapoint, key, sample_variance=sample_variance
        )
        dataloader = dataloader.add(next_datapoint, next_target)

        # 2) Continue training
        model = train_model(model, dataloader, n_epochs=epochs_per_learning_round)

        # 3) Calibrate and compute uncertainty
        posterior_fn = get_posterior_fn(model, dataloader)
        grid_params = {
            "current_guess": prior_prec,
            "magnitudes_to_search": 2,
            "grid_size": 25,
        }
        prior_prec = calibrate_prior_precision(
            dataloader, model, posterior_fn, grid_params
        )
        print(f"Calibrated precision: {prior_prec:.0f}")
        kernel = get_posterior_covariance_kernel(model, posterior_fn, prior_prec)

        # 4) Find next datapoint location
        next_datapoint = maximize_total_information_gain(kernel, x_pred)

        # Plotting
        y_mean = model(x_pred)
        y_std = get_uncertainty_from_kernel(kernel, x_pred)
        plot_data.append((x_pred, y_mean, y_true, dataloader, y_std, next_datapoint))
        print("-----------------------")

    loguru.logger.enable("laplax.eval.calibrate")
    return plot_data


plot_data = active_learning_loop(
    model, next_datapoint, dataloader, prior_prec, learning_rounds
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))


def update(frame):
    ax.clear()
    artists1 = plot_data_and_uncertainty_around_prediction(ax, *(plot_data[frame]))
    return artists1


animation = FuncAnimation(
    fig, update, frames=learning_rounds, interval=1500, repeat_delay=2000
)
plt.close(fig)  # Prevent duplicate figure
HTML(animation.to_jshtml())

## Comparison to passive learning

To see the difference active learning makes, we compare the learned model to one that is passively trained, i.e. one where the datapoints are not chosen smartly.

For a fair comparison, we train the passive model with the same number of datapoints and for the same overall number of epochs.\
Note however that in active learning, epochs are much smaller in the beginning.
You can choose between sampling the datapoints randomly (uniform) or with deterministic equidistant spacing.

In [ ]:
sampling_dropdown = widgets.Dropdown(
    options=["Random Uniform", "Equidistant"],
    value="Random Uniform",
    description="Sampling:",
)
display(sampling_dropdown)

In [ ]:
n_passive_datapoints = n_initial_datapoints + learning_rounds
n_passive_epochs = n_initial_epochs + learning_rounds * epochs_per_learning_round

# Sample x-values according to selection
sampling_type = sampling_dropdown.value
random_uniform = random.uniform(key, shape=(20, 1), minval=0.2, maxval=5.0)
equidistant = jnp.linspace(0.2, 5, 20)[:, None]
passive_xs = random_uniform if sampling_type == "Random Uniform" else equidistant

# Sample y-values
keys = random.split(key, len(passive_xs))
passive_ys = vmap(sample)(passive_xs, keys)[:, None]

# Train model with sampled data
passive_dataloader = DataLoader(passive_xs, passive_ys, batch_size=10)
if len(passive_dataloader) != len(dataloader):
    print("Number of datapoints for active and passive learning do not match!")

passive_model = Model(
    in_channels=1, hidden_channels=64, out_channels=1, rngs=nnx.Rngs(seed)
)
passive_model = train_model(
    passive_model, passive_dataloader, n_epochs=n_passive_epochs
)

# Predict with passive model
y_pred_passive = passive_model(x_pred)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
plot_model_comparison(ax, x_pred, y_true, y_pred_passive, y_mean, passive_dataloader)
plt.show()

# Compute RMSE to exact function
passive_rmse = jnp.sqrt(jnp.mean((y_pred_passive - y_true) ** 2))
active_rmse = jnp.sqrt(jnp.mean((y_mean - y_true) ** 2))

print(f"RMSE of passive model to true function: {passive_rmse:.2f}")
print(f"RMSE of active model to true function: {active_rmse:.2f}")

As we can see (at least in the standard configuration), the actively learned network exhibits a significantly lower RMSE than the passively learned one.

## Maximizing information about a point of interest

In [ ]:
def maximize_information_gain_about_point(kernel, x_pred, point):
    """Find point where the total information gain is maximal.

    Args:
        kernel: Posterior covariance kernel of the model
        x_pred: Candidate points
        point: Point of interest where information should be maximized

    Returns:
        Point from x_pred that gives the most information about the point of interest.
    """
    variance_u = kernel([point], [point])
    variance_nu = 1.0 / prior_prec
    variances_x = kernel(x_pred, x_pred)
    covariance_xu = kernel(x_pred, [point])
    criterium = covariance_xu**2 / (variance_u * (variance_nu + variances_x))
    return find_maximum(x_pred, criterium), criterium

In [ ]:
y_mean = model(x_pred)
kernel = get_posterior_covariance_kernel(model, posterior_fn, prior_prec)
point, crit = maximize_information_gain_about_point(kernel, x_pred, 3.0)

fig, ax = plt.subplots(figsize=(10, 5))
plot_data_and_uncertainty_around_prediction(
    ax, x_pred, y_mean, y_true, dataloader, crit
)
ax.set_ylim((-2, 2))
ax.axvline(point)
plt.show()